# Blog 6 — Delta Lake MERGE: How Upserts Actually Work

**Databricks + PySpark + SQL**

A production-minded notebook for understanding and implementing Delta Lake `MERGE`.

## Environment

This notebook is built for the verified Databricks environment:

- **Catalog:** `workspace`
- **Schema:** `medallion_project`
- **Target:** `workspace.medallion_project.customers_merge_demo`

It uses Unity Catalog managed Delta tables and does not depend on DBFS root paths.

## What this notebook covers

1. The upsert problem
2. `MERGE` anatomy
3. Basic update + insert
4. Duplicate source records
5. Deterministic deduplication **and an actual MERGE**
6. Conditional updates
7. Deletes
8. Incremental processing
9. Idempotency with a real retry test
10. A source-operation footgun and validation
11. Performance considerations
12. Common mistakes
13. A clean reusable production pattern
14. SQL equivalent
15. Assertions and final validation


## 1. Environment Setup

The notebook uses the catalog and schema that were verified in the Databricks workspace.

```text
workspace
└── medallion_project
```

We deliberately use a managed table rather than a filesystem path.


In [0]:
# Fixed environment for this notebook

catalog = "workspace"
schema = "medallion_project"
target_table = f"{catalog}.{schema}.customers_merge_demo"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

print("Catalog :", catalog)
print("Schema  :", schema)
print("Target  :", target_table)


Catalog : workspace
Schema  : medallion_project
Target  : workspace.medallion_project.customers_merge_demo


## 2. Imports


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable


## 3. Test Helpers

This notebook intentionally uses assertions instead of relying only on `display()`.

That gives us a repeatable rule:

> If a section produces the wrong state, the notebook should fail loudly.

The helpers below let each conceptual demo:

- start from a clean baseline
- execute its own MERGE
- validate row counts
- validate key uniqueness
- validate important business values


In [0]:
def reset_target(rows):
    """Recreate the managed Delta target from a deterministic list of rows."""
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")

    df = (
        spark.createDataFrame(
            rows,
            ["customer_id", "name", "city", "updated_at"]
        )
        .withColumn("updated_at", F.to_timestamp("updated_at"))
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    return DeltaTable.forName(spark, target_table)


def assert_row_count(expected):
    actual = spark.table(target_table).count()
    assert actual == expected, (
        f"Row-count check failed: expected {expected}, got {actual}"
    )


def assert_unique_customer_keys():
    duplicate_count = (
        spark.table(target_table)
        .groupBy("customer_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicate_count == 0, (
        f"Duplicate customer_id keys detected: {duplicate_count}"
    )


def assert_customer(customer_id, expected_city, expected_updated_at=None):
    row = (
        spark.table(target_table)
        .filter(F.col("customer_id") == customer_id)
        .first()
    )

    assert row is not None, f"Customer {customer_id} was not found"
    assert row["city"] == expected_city, (
        f"Customer {customer_id}: expected city "
        f"{expected_city}, got {row['city']}"
    )

    if expected_updated_at is not None:
        expected_ts = spark.sql(
            f"SELECT to_timestamp('{expected_updated_at}') AS ts"
        ).first()["ts"]

        assert row["updated_at"] == expected_ts, (
            f"Customer {customer_id}: expected updated_at "
            f"{expected_ts}, got {row['updated_at']}"
        )


def assert_customer_absent(customer_id):
    exists = (
        spark.table(target_table)
        .filter(F.col("customer_id") == customer_id)
        .limit(1)
        .count()
    )

    assert exists == 0, f"Customer {customer_id} should not exist"


def show_target():
    display(
        spark.table(target_table)
        .orderBy("customer_id")
    )


## 4. The Upsert Problem

Suppose the target contains:

| customer_id | name | city |
|---:|---|---|
| 1 | Arun | Chennai |
| 2 | Ravi | Madurai |
| 3 | Kumar | Salem |

A new batch contains:

| customer_id | name | city |
|---:|---|---|
| 2 | Ravi | Coimbatore |
| 4 | Priya | Chennai |

The desired result is:

| customer_id | name | city |
|---:|---|---|
| 1 | Arun | Chennai |
| 2 | Ravi | Coimbatore |
| 3 | Kumar | Salem |
| 4 | Priya | Chennai |

Customer 2 is an **update**.

Customer 4 is an **insert**.

This combined operation is an **upsert**.


## 5. `MERGE` Anatomy

The core matching rule is:

```text
target.customer_id = source.customer_id
```

Conceptually:

```text
                SOURCE
                   |
                   v
             Match condition
                   |
          +--------+--------+
          |                 |
       MATCHED          NOT MATCHED
          |                 |
          v                 v
        UPDATE            INSERT
```

A Delta Lake `MERGE` lets us express these decisions declaratively.


## 6. Demo 1 — Basic UPDATE + INSERT

### Start from a clean baseline

Every major demonstration in this notebook resets the target first.

This prevents one demo from silently changing the starting state of another.


In [0]:
target = reset_target([
    (1, "Arun", "Chennai", "2026-08-18 09:00:00"),
    (2, "Ravi", "Madurai", "2026-08-18 09:00:00"),
    (3, "Kumar", "Salem", "2026-08-18 09:00:00")
])

assert_row_count(3)
assert_unique_customer_keys()

show_target()


customer_id,name,city,updated_at
1,Arun,Chennai,2026-08-18T09:00:00.000Z
2,Ravi,Madurai,2026-08-18T09:00:00.000Z
3,Kumar,Salem,2026-08-18T09:00:00.000Z


### Create the source

For this basic demonstration, the source schema intentionally matches the target schema. We do not introduce an `operation` column yet.

This keeps `UPDATE SET *` and `INSERT *` unambiguous.


In [0]:
basic_source = (
    spark.createDataFrame(
        [
            (2, "Ravi", "Coimbatore", "2026-08-19 10:00:00"),
            (4, "Priya", "Chennai", "2026-08-19 12:00:00")
        ],
        ["customer_id", "name", "city", "updated_at"]
    )
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

display(basic_source)


customer_id,name,city,updated_at
2,Ravi,Coimbatore,2026-08-19T10:00:00.000Z
4,Priya,Chennai,2026-08-19T12:00:00.000Z


In [0]:
(
    target.alias("target")
    .merge(
        basic_source.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

assert_row_count(4)
assert_unique_customer_keys()
assert_customer(2, "Coimbatore", "2026-08-19 10:00:00")
assert_customer(4, "Chennai", "2026-08-19 12:00:00")

show_target()

print("PASS — basic UPDATE + INSERT")


customer_id,name,city,updated_at
1,Arun,Chennai,2026-08-18T09:00:00.000Z
2,Ravi,Coimbatore,2026-08-19T10:00:00.000Z
3,Kumar,Salem,2026-08-18T09:00:00.000Z
4,Priya,Chennai,2026-08-19T12:00:00.000Z


PASS — basic UPDATE + INSERT


## 7. Demo 2 — Duplicate Source Records

A source can contain more than one record for the same business key:

```text
customer_id | city        | updated_at
------------|-------------|-------------------
2           | Coimbatore  | 10:00
2           | Trichy      | 11:00
```

Before a `MERGE`, we need a deterministic rule.

For this notebook:

> Keep the latest record by `updated_at`.


In [0]:
duplicate_source = (
    spark.createDataFrame(
        [
            (2, "Ravi", "Coimbatore", "2026-08-19 10:00:00"),
            (2, "Ravi", "Trichy", "2026-08-19 11:00:00"),
            (4, "Priya", "Chennai", "2026-08-19 12:00:00")
        ],
        ["customer_id", "name", "city", "updated_at"]
    )
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

display(
    duplicate_source
    .orderBy("customer_id", "updated_at")
)


customer_id,name,city,updated_at
2,Ravi,Coimbatore,2026-08-19T10:00:00.000Z
2,Ravi,Trichy,2026-08-19T11:00:00.000Z
4,Priya,Chennai,2026-08-19T12:00:00.000Z


## 8. Demo 2 Continued — Deduplicate AND Actually MERGE

This is intentionally an end-to-end test.

We will:

1. reset the target
2. deduplicate the source
3. merge the deduplicated source
4. assert that customer 2 became `Trichy`
5. assert that customer 4 was inserted


In [0]:
target = reset_target([
    (1, "Arun", "Chennai", "2026-08-18 09:00:00"),
    (2, "Ravi", "Madurai", "2026-08-18 09:00:00"),
    (3, "Kumar", "Salem", "2026-08-18 09:00:00")
])

window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("updated_at").desc())
)

deduplicated_source = (
    duplicate_source
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(
    deduplicated_source
    .orderBy("customer_id")
)

# Prove the deduplication actually produced one row per key.
duplicate_keys_in_source = (
    deduplicated_source
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_keys_in_source == 0, (
    f"Deduplication failed: {duplicate_keys_in_source} duplicate keys remain"
)


customer_id,name,city,updated_at
2,Ravi,Trichy,2026-08-19T11:00:00.000Z
4,Priya,Chennai,2026-08-19T12:00:00.000Z


In [0]:
(
    target.alias("target")
    .merge(
        deduplicated_source.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

assert_row_count(4)
assert_unique_customer_keys()
assert_customer(2, "Trichy", "2026-08-19 11:00:00")
assert_customer(4, "Chennai", "2026-08-19 12:00:00")

show_target()

print("PASS — deduplication was applied successfully through MERGE")


customer_id,name,city,updated_at
1,Arun,Chennai,2026-08-18T09:00:00.000Z
2,Ravi,Trichy,2026-08-19T11:00:00.000Z
3,Kumar,Salem,2026-08-18T09:00:00.000Z
4,Priya,Chennai,2026-08-19T12:00:00.000Z


PASS — deduplication was applied successfully through MERGE


## 9. Demo 3 — Conditional Updates

A matched record should not always be updated.

Suppose:

```text
Target updated_at = 11:00
Source updated_at = 09:00
```

The source is older.

We want:

```text
source.updated_at > target.updated_at
```

to be required before an update occurs.


In [0]:
target = reset_target([
    (1, "Arun", "Chennai", "2026-08-18 09:00:00"),
    (2, "Ravi", "Madurai", "2026-08-19 11:00:00"),
    (3, "Kumar", "Salem", "2026-08-18 09:00:00")
])

older_source = (
    spark.createDataFrame(
        [
            (2, "Ravi", "Old City", "2026-08-19 09:00:00")
        ],
        ["customer_id", "name", "city", "updated_at"]
    )
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

(
    target.alias("target")
    .merge(
        older_source.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll(
        condition="source.updated_at > target.updated_at"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

assert_row_count(3)
assert_customer(2, "Madurai", "2026-08-19 11:00:00")

show_target()

print("PASS — older source record did not overwrite newer target data")


customer_id,name,city,updated_at
1,Arun,Chennai,2026-08-18T09:00:00.000Z
2,Ravi,Madurai,2026-08-19T11:00:00.000Z
3,Kumar,Salem,2026-08-18T09:00:00.000Z


PASS — older source record did not overwrite newer target data


## 10. Demo 4 — Newer Conditional UPDATE

Now test the opposite case.

The source is newer than the target, so the update should happen.


In [0]:
target = reset_target([
    (1, "Arun", "Chennai", "2026-08-18 09:00:00"),
    (2, "Ravi", "Madurai", "2026-08-19 11:00:00"),
    (3, "Kumar", "Salem", "2026-08-18 09:00:00")
])

newer_source = (
    spark.createDataFrame(
        [
            (2, "Ravi", "Coimbatore", "2026-08-19 12:00:00")
        ],
        ["customer_id", "name", "city", "updated_at"]
    )
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

(
    target.alias("target")
    .merge(
        newer_source.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll(
        condition="source.updated_at > target.updated_at"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

assert_customer(2, "Coimbatore", "2026-08-19 12:00:00")

print("PASS — newer source record updated the target")


PASS — newer source record updated the target


## 11. Demo 5 — DELETE

Incremental systems can also emit delete events.

For this demonstration we use an `operation` column in the source and explicit column mappings.

We do **not** use `UpdateAll`/`InsertAll` here because the source contains an extra `operation` field that is not part of the target table.


In [0]:
target = reset_target([
    (1, "Arun", "Chennai", "2026-08-18 09:00:00"),
    (2, "Ravi", "Madurai", "2026-08-19 11:00:00"),
    (3, "Kumar", "Salem", "2026-08-18 09:00:00")
])

delete_source = (
    spark.createDataFrame(
        [
            (3, "Kumar", "Salem", "2026-08-19 13:00:00", "DELETE")
        ],
        ["customer_id", "name", "city", "updated_at", "operation"]
    )
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

(
    target.alias("target")
    .merge(
        delete_source.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedDelete(
        condition="source.operation = 'DELETE'"
    )
    .execute()
)

assert_row_count(2)
assert_unique_customer_keys()
assert_customer_absent(3)

show_target()

print("PASS — DELETE event removed the target record")


customer_id,name,city,updated_at
1,Arun,Chennai,2026-08-18T09:00:00.000Z
2,Ravi,Madurai,2026-08-19T11:00:00.000Z


PASS — DELETE event removed the target record


## 12. Soft Delete Alternative

Physical deletion is not always appropriate.

Another design is to retain the row and update a flag:

```text
customer_id | is_deleted
------------|-----------
3           | true
```

That preserves the historical row while allowing downstream logic to exclude deleted records.

The choice is a business/data-model decision.


## 13. Incremental Processing + MERGE

Incremental processing answers:

> **Which records changed?**

`MERGE` answers:

> **How should those changes be applied to the target?**

The production flow is:

```text
Source System
     |
     v
Identify Changes
     |
     v
Validate
     |
     v
Deduplicate
     |
     v
MERGE
 |      |       |
 v      v       v
INSERT UPDATE DELETE
     |
     v
Delta Target
```

The notebook's tests above deliberately keep the target baseline isolated so that each concept can be verified independently.


## 14. Demo 6 — Idempotency Retry Test

A pipeline should be safe to retry.

The test:

1. reset the target
2. apply the same incremental batch
3. capture the result
4. apply the **same batch again**
5. capture the result again
6. assert that the final states are identical

This turns the idempotency explanation into an executable test.


In [0]:
target = reset_target([
    (1, "Arun", "Chennai", "2026-08-18 09:00:00"),
    (2, "Ravi", "Madurai", "2026-08-18 09:00:00"),
    (3, "Kumar", "Salem", "2026-08-18 09:00:00")
])

idempotent_source = (
    spark.createDataFrame(
        [
            (2, "Ravi", "Coimbatore", "2026-08-19 10:00:00"),
            (4, "Priya", "Chennai", "2026-08-19 12:00:00")
        ],
        ["customer_id", "name", "city", "updated_at"]
    )
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)


In [0]:
# First run
(
    target.alias("target")
    .merge(
        idempotent_source.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll(
        condition="source.updated_at > target.updated_at"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

first_state = (
    spark.table(target_table)
    .orderBy("customer_id")
    .collect()
)

assert_row_count(4)
assert_unique_customer_keys()
assert_customer(2, "Coimbatore", "2026-08-19 10:00:00")
assert_customer(4, "Chennai", "2026-08-19 12:00:00")

print("First run: PASS")


First run: PASS


In [0]:
# Retry: run the exact same source again
(
    target.alias("target")
    .merge(
        idempotent_source.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll(
        condition="source.updated_at > target.updated_at"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

second_state = (
    spark.table(target_table)
    .orderBy("customer_id")
    .collect()
)

assert first_state == second_state, (
    "Idempotency check failed: the second identical run changed the final state"
)

assert_row_count(4)
assert_unique_customer_keys()

print("Second run: PASS")
print("PASS — rerunning the same batch produced the same final state")


Second run: PASS
PASS — rerunning the same batch produced the same final state


## 15. A Realistic Footgun — Invalid `operation` Values

Consider this pattern:

```python
.whenNotMatchedInsertAll(
    condition="source.operation = 'INSERT'"
)
```

What happens if a genuinely new key arrives but the upstream source incorrectly labels it:

```text
operation = 'UPDATE'
```

The row is:

- not matched
- not eligible for INSERT
- not eligible for UPDATE

So it can silently produce **no target change**.

That is a data-quality problem, not a Delta Lake feature.

We should detect invalid or contradictory source records before the `MERGE`.


In [0]:
target = reset_target([
    (1, "Arun", "Chennai", "2026-08-18 09:00:00"),
    (2, "Ravi", "Madurai", "2026-08-19 11:00:00"),
    (3, "Kumar", "Salem", "2026-08-18 09:00:00")
])

bad_new_key = (
    spark.createDataFrame(
        [
            (99, "New Customer", "Chennai", "2026-08-19 14:00:00", "UPDATE")
        ],
        ["customer_id", "name", "city", "updated_at", "operation"]
    )
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

# Demonstrate the footgun.
(
    target.alias("target")
    .merge(
        bad_new_key.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedDelete(
        condition="source.operation = 'DELETE'"
    )
    .whenMatchedUpdate(
        condition=(
            "source.operation = 'UPDATE' "
            "AND source.updated_at > target.updated_at"
        ),
        set={
            "name": "source.name",
            "city": "source.city",
            "updated_at": "source.updated_at"
        }
    )
    .whenNotMatchedInsert(
        condition="source.operation = 'INSERT'",
        values={
            "customer_id": "source.customer_id",
            "name": "source.name",
            "city": "source.city",
            "updated_at": "source.updated_at"
        }
    )
    .execute()
)

assert_customer_absent(99)

print(
    "Expected footgun demonstrated: customer 99 was not inserted "
    "because the source incorrectly labelled a new key as UPDATE."
)


Expected footgun demonstrated: customer 99 was not inserted because the source incorrectly labelled a new key as UPDATE.


### Prevent the footgun with preflight validation

A robust pipeline should validate source operations before the merge.

At minimum:

```text
operation must be one of:
INSERT, UPDATE, DELETE
```

For a new key labelled `UPDATE`, the pipeline can also detect the mismatch by comparing source keys with the target before executing the merge.



In [0]:
allowed_operations = {"INSERT", "UPDATE", "DELETE"}

invalid_operations = (
    bad_new_key
    .filter(~F.col("operation").isin(list(allowed_operations)))
    .count()
)

assert invalid_operations == 0, (
    f"Invalid operation values detected: {invalid_operations}"
)

# Detect source rows marked UPDATE/DELETE whose keys do not exist in the target.
target_keys = spark.table(target_table).select("customer_id").distinct()

invalid_new_key_operations = (
    bad_new_key.alias("source")
    .join(
        target_keys.alias("target"),
        on="customer_id",
        how="left_anti"
    )
    .filter(F.col("operation").isin("UPDATE", "DELETE"))
)

invalid_count = invalid_new_key_operations.count()

assert invalid_count == 1, (
    "Expected the intentionally bad new-key UPDATE to be detected"
)

print("PASS — preflight validation detected the invalid new-key UPDATE")


PASS — preflight validation detected the invalid new-key UPDATE


## 16. Correct Operation-Aware MERGE Pattern

When the source contains an `operation` column that is not part of the target schema, prefer explicit mappings.

This makes the target columns obvious and prevents accidental schema coupling.


In [0]:
# Reset to a clean baseline
target = reset_target([
    (1, "Arun", "Chennai", "2026-08-18 09:00:00"),
    (2, "Ravi", "Madurai", "2026-08-18 09:00:00"),
    (3, "Kumar", "Salem", "2026-08-18 09:00:00")
])

operation_source = (
    spark.createDataFrame(
        [
            (2, "Ravi", "Coimbatore", "2026-08-19 10:00:00", "UPDATE"),
            (4, "Priya", "Chennai", "2026-08-19 12:00:00", "INSERT"),
            (3, "Kumar", "Salem", "2026-08-19 13:00:00", "DELETE")
        ],
        ["customer_id", "name", "city", "updated_at", "operation"]
    )
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

# Deduplicate by key and latest timestamp.
operation_deduped = (
    operation_source
    .withColumn("rn", F.row_number().over(
        Window
        .partitionBy("customer_id")
        .orderBy(F.col("updated_at").desc())
    ))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

(
    target.alias("target")
    .merge(
        operation_deduped.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedDelete(
        condition="source.operation = 'DELETE'"
    )
    .whenMatchedUpdate(
        condition=(
            "source.operation = 'UPDATE' "
            "AND source.updated_at > target.updated_at"
        ),
        set={
            "name": "source.name",
            "city": "source.city",
            "updated_at": "source.updated_at"
        }
    )
    .whenNotMatchedInsert(
        condition="source.operation = 'INSERT'",
        values={
            "customer_id": "source.customer_id",
            "name": "source.name",
            "city": "source.city",
            "updated_at": "source.updated_at"
        }
    )
    .execute()
)

assert_row_count(3)
assert_unique_customer_keys()
assert_customer(2, "Coimbatore", "2026-08-19 10:00:00")
assert_customer(4, "Chennai", "2026-08-19 12:00:00")
assert_customer_absent(3)

show_target()

print("PASS — INSERT + UPDATE + DELETE operation-aware MERGE")


customer_id,name,city,updated_at
1,Arun,Chennai,2026-08-18T09:00:00.000Z
2,Ravi,Coimbatore,2026-08-19T10:00:00.000Z
4,Priya,Chennai,2026-08-19T12:00:00.000Z


PASS — INSERT + UPDATE + DELETE operation-aware MERGE


## 17. MERGE Performance Considerations

A logically correct `MERGE` can still be inefficient.

### Reduce the source before MERGE

Filter irrelevant records and remove duplicates first.

### Avoid unnecessary updates

Use a version/timestamp condition when that reflects the business rule.

### Choose the correct merge key

The `ON` condition must match the business grain.

If the target grain is:

```text
customer_id + product_id
```

matching only on `customer_id` would be incorrect.

### Think about table layout

Do not automatically partition by a high-cardinality merge key. Optimization depends on workload, data distribution, table size, and runtime.

> **Optimize the entire data flow leading into `MERGE`, not just the `MERGE` statement.**


## 18. Common MERGE Mistakes

1. **Not deduplicating the source** — multiple source rows for one merge key create ambiguity.
2. **Using the wrong merge key** — the `ON` condition must match the business grain.
3. **Blindly updating every match** — stale data can overwrite newer target data.
4. **Assuming MERGE alone guarantees idempotency** — the entire pipeline must be deterministic.
5. **Processing unnecessary source rows** — reduce the incremental source before the merge.
6. **Ignoring late-arriving data** — use a deterministic version/timestamp rule.
7. **Using `UpdateAll`/`InsertAll` when the source has extra non-target columns** — use explicit mappings when schemas differ.
8. **Allowing invalid operation flags** — a new key incorrectly labelled `UPDATE` can be silently skipped by a conditional insert.
9. **Sharing mutable target state across unrelated demos** — reset the target before each conceptual test.
10. **Relying only on `display()`** — use assertions to turn incorrect states into explicit failures.


## 19. Clean Reusable Production Pattern

The reusable logic should be independent of the earlier demonstrations.

The pattern is:

```text
Incremental Source
       |
       v
Validate Source
       |
       v
Deduplicate by Business Key
       |
       v
Check Operation / Business Rules
       |
       v
MERGE
   |       |       |
 INSERT  UPDATE  DELETE
       |
       v
Delta Target
       |
       v
Assertions / Data Quality Checks
```

In a real pipeline, the incremental source would come from a source system, CDC feed, Auto Loader, or another ingestion process.

For this notebook, the source DataFrame is supplied directly so the notebook remains self-contained and executable.


In [0]:
# Reusable production-style example.
# In this notebook, `production_source` is an in-memory incremental batch.

target = reset_target([
    (1, "Arun", "Chennai", "2026-08-18 09:00:00"),
    (2, "Ravi", "Madurai", "2026-08-18 09:00:00"),
    (3, "Kumar", "Salem", "2026-08-18 09:00:00")
])

production_source = (
    spark.createDataFrame(
        [
            (2, "Ravi", "Coimbatore", "2026-08-19 10:00:00", "UPDATE"),
            (4, "Priya", "Chennai", "2026-08-19 12:00:00", "INSERT")
        ],
        ["customer_id", "name", "city", "updated_at", "operation"]
    )
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

# Validate operation values.
invalid_operation_count = (
    production_source
    .filter(~F.col("operation").isin("INSERT", "UPDATE", "DELETE"))
    .count()
)

assert invalid_operation_count == 0, (
    f"Invalid operation values: {invalid_operation_count}"
)

# Deduplicate source.
production_source_deduped = (
    production_source
    .withColumn(
        "rn",
        F.row_number().over(
            Window
            .partitionBy("customer_id")
            .orderBy(F.col("updated_at").desc())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Merge using explicit target mappings.
(
    target.alias("target")
    .merge(
        production_source_deduped.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedDelete(
        condition="source.operation = 'DELETE'"
    )
    .whenMatchedUpdate(
        condition=(
            "source.operation = 'UPDATE' "
            "AND source.updated_at > target.updated_at"
        ),
        set={
            "name": "source.name",
            "city": "source.city",
            "updated_at": "source.updated_at"
        }
    )
    .whenNotMatchedInsert(
        condition="source.operation = 'INSERT'",
        values={
            "customer_id": "source.customer_id",
            "name": "source.name",
            "city": "source.city",
            "updated_at": "source.updated_at"
        }
    )
    .execute()
)

# Validate the resulting target.
assert_row_count(4)
assert_unique_customer_keys()
assert_customer(2, "Coimbatore", "2026-08-19 10:00:00")
assert_customer(4, "Chennai", "2026-08-19 12:00:00")

show_target()

print("PASS — reusable production-style pattern")


customer_id,name,city,updated_at
1,Arun,Chennai,2026-08-18T09:00:00.000Z
2,Ravi,Coimbatore,2026-08-19T10:00:00.000Z
3,Kumar,Salem,2026-08-18T09:00:00.000Z
4,Priya,Chennai,2026-08-19T12:00:00.000Z


PASS — reusable production-style pattern


## 20. SQL Equivalent

The operation-aware logic can be expressed in SQL:

```sql
MERGE INTO workspace.medallion_project.customers_merge_demo AS target
USING customer_changes AS source
ON target.customer_id = source.customer_id

WHEN MATCHED
  AND source.operation = 'DELETE'
THEN DELETE

WHEN MATCHED
  AND source.operation = 'UPDATE'
  AND source.updated_at > target.updated_at
THEN UPDATE SET
    target.name = source.name,
    target.city = source.city,
    target.updated_at = source.updated_at

WHEN NOT MATCHED
  AND source.operation = 'INSERT'
THEN INSERT (
    customer_id,
    name,
    city,
    updated_at
)
VALUES (
    source.customer_id,
    source.name,
    source.city,
    source.updated_at
);
```

The explicit column mappings make the target/source schema relationship clear.


## 21. What Actually Happens During a MERGE?

Conceptually:

```text
          Source Data
               |
               v
        Evaluate MERGE condition
               |
       +-------+-------+
       |               |
    MATCHED        NOT MATCHED
       |               |
       v               v
Evaluate clauses    INSERT clauses
       |
   +---+-------+
   |           |
 UPDATE       DELETE
       |
       v
  New Table State
```

The useful mental model is:

> **`MERGE` is a conditional synchronization operation between a source dataset and a target Delta table.**

The difficult engineering work is deciding:

- what identifies a record
- which source record wins when duplicates exist
- when an update is valid
- how deletes are represented
- how invalid source events are detected
- how retries behave


## 22. Final Full-State Validation

This is the final safety check for the reusable production-style example.

We validate:

- expected row count
- unique business keys
- expected updated customer
- expected inserted customer


In [0]:
assert_row_count(4)
assert_unique_customer_keys()
assert_customer(2, "Coimbatore", "2026-08-19 10:00:00")
assert_customer(4, "Chennai", "2026-08-19 12:00:00")

print("========================================")
print("ALL FINAL ASSERTIONS PASSED")
print("========================================")


ALL FINAL ASSERTIONS PASSED


## 23. Key Takeaways

1. **`MERGE` is a synchronization operation**, not simply an `UPDATE` plus `INSERT`.
2. **The `ON` condition is critical** because it defines how records correspond.
3. **Deduplicate the source before merging** so each merge key has a deterministic record.
4. **The deduplicated source must actually be used in the MERGE** — demonstrating it with `display()` is not enough.
5. **Conditional updates** protect against stale or late-arriving records.
6. **`MERGE` can handle inserts, updates, and deletes.**
7. **Idempotency should be tested by rerunning the same batch**, not just described conceptually.
8. **Operation-aware MERGE logic should use explicit mappings** when the source contains columns not present in the target.
9. **Invalid source operations can cause silent data loss** when conditional clauses exclude a record from every action.
10. **Each major demonstration should start from a deterministic baseline.**
11. **Assertions make notebook examples executable tests rather than visual demonstrations.**
12. **Performance starts before the merge** by reducing, validating, and deduplicating the source.

### Final takeaway

> **Delta Lake `MERGE` turns incremental changes into a controlled synchronization process. The real engineering challenge is not learning the syntax — it is designing deterministic source preparation, matching logic, update rules, delete behavior, validation, and retry semantics correctly.**


## 24. Optional Cleanup

Run this only if you want to remove the demo table after completing the notebook.

```python
spark.sql(f"DROP TABLE IF EXISTS {target_table}")
```
